# Step 1: Install and Import Libraries

In [1]:
!pip install -q tensorflow numpy

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from IPython.display import display, HTML


# Step 2: Sample Dataset (Movie Reviews)

In [2]:

reviews = [
    # Positive
    "Amazing movie! Loved the visuals and acting.",
    "A beautiful story told well. Highly recommended!",
    "Fantastic experience. The direction was top-notch.",

    # Neutral
    "The movie was okay, not great but not terrible.",
    "Average plot and performances. Nothing stood out.",
    "Some parts were good, others were slow.",

    # Negative
    "Terrible film. Poor acting and weak plot.",
    "Very disappointing. I expected a lot more.",
    "A complete waste of time. Would not recommend."
]

labels = [0, 0, 0,    # Positive
          1, 1, 1,    # Neutral
          2, 2, 2]    # Negative

# Step 3: Text Preprocessing

In [3]:
num_classes = 3
num_words = 1000
max_length = 20

# Tokenize text
tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(reviews)
sequences = tokenizer.texts_to_sequences(reviews)
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, labels, test_size=0.2, random_state=42)
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

# Step 4: Define and Train the Model

In [4]:

model = Sequential([
    Embedding(num_words, 16, input_length=max_length),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()
model.fit(X_train, y_train, epochs=30, validation_data=(X_test, y_test), verbose=0)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ ?                           │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Step 5: Prediction UI Function

In [5]:

def predict_sentiment(text):
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=max_length, padding='post')
    prediction = model.predict(padded)
    class_names = ['Positive', 'Neutral', 'Negative']
    predicted_class = np.argmax(prediction)
    confidence = np.max(prediction)

    display(HTML(f"""
    <div style='font-size:24px; font-weight:bold; color:green;'>
        Sentiment Prediction: {class_names[predicted_class]}<br>
        Confidence: {confidence:.2f}
    </div>
    """))

# Step 6: Try with User Input

In [8]:

user_input = input("Enter a movie review: ")
predict_sentiment(user_input)

Enter a movie review: It was fine. Not particularly memorable, but not bad either.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step


# Save the Model

In [11]:
# Save the trained model to an HDF5 file
model.save("TextClassifierModel.h5")
print("✅ Model saved as 'TextClassifierModel.h5'")


✅ Model saved as 'TextClassifierModel.h5'


# Download the Model

In [13]:
# Download the saved model to your local machine
from google.colab import files
files.download("TextClassifierModel.h5")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Reload

In [ ]:
from tensorflow.keras.models import load_model
model = load_model("sentiment_model.h5")
